# 04. Lecke - Eszközhasználati tervezési minta

Ebben a leckében megismerkedhet az AI ügynökök számára a Microsoft Agent Framework (Python) alapján készült **Eszközhasználat** tervezési mintával. A következőket tárgyaljuk:

- Funkcióeszközök definiálása az `@tool` dekorátorral és típusos paraméterekkel
- Eszközsémák megadása, hogy a modell tudja, mit csinál az egyes eszközök
- Az eszközvégrehajtás vezérlése az `approval_mode` segítségével
- **Strukturált kimenet** visszaadása Pydantic modellek és a `response_format` használatával

A forgatókönyv egy **utazási foglaló ügynök**, amely képes úti célokat keresni, elérhetőséget ellenőrizni és repülőjárat-információkat lekérni.


## Beállítás


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Az eszközök definiálása az @tool dekorátorral

Az `@tool` dekorátor egy egyszerű Python függvényt alakít át olyan eszközzé, amelyet egy ügynök meghívhat.  
Kulcspontok:

- A **docstring** lesz az eszköz leírása, amelyet a modell lát.  
- A **típusannotációk** (beleértve az `Annotated`-ot leírásokkal) definiálják az eszköz sémáját.  
- Az `approval_mode` vezérli, hogy a felhasználónak jóvá kell-e hagynia minden meghívást, mielőtt az végrehajtódik.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## Ügynök létrehozása több eszközzel

Add át mindhárom eszközt az ügyfélnek, hogy a modell bármelyiket meghívhassa, amelyikre szüksége van a felhasználó kérdésének megválaszolásához.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## Strukturált kimenet eszközökkel

A `response_format` Pydantic modellre állításával a ügynök arra van kényszerítve, hogy jól típusos JSON objektumot adjon vissza szabad formátumú szöveg helyett. Ez akkor hasznos, ha a további feldolgozó kódnak programozottan kell felhasználnia az eredményt.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## Eszköz jóváhagyási minták

Az `@tool` `approval_mode` paramétere szabályozza, hogy az eszköz hívások végrehajtása előtt szükséges-e emberi jóváhagyás:

| Mód | Viselkedés |
|---|---|
| `"never_require"` | Az eszköz automatikusan fut — nincs szükség felhasználói megerősítésre. |
| `"always_require"` | Minden hívást a felhasználónak jóvá kell hagynia, mielőtt végrehajtásra kerül. |

Az olyan eszközöknél használja az `"always_require"` módot, amelyek mellékhatásokkal járnak (pl. repülőjegy foglalása, hitelkártya terhelése), hogy az ember mindig beleszólhasson a folyamatba.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## Összefoglaló

Ebben a leckében megtanultad, hogyan:

1. **Definiálj eszközöket** az `@tool` dekorátor segítségével, típusos paraméterekkel és docstringekkel, amelyek az eszköz séma szerepét töltik be.
2. **Több eszközt összekapcsolj**, hogy az ügynök egymás után hívhassa meg őket összetett lekérdezések megválaszolásához.
3. **Szervezett kimenetet adj vissza** egy Pydantic modell átadásával `response_format`ként.
4. **Szabályozd az eszköz jóváhagyását** az `approval_mode` segítségével, hogy érzékeny műveletek esetén emberi láncban maradjon.

Ezek a minták képezik az alapját a megbízható, éles környezetbe szánt ügynökök építésének, amelyek biztonságosan tudnak külső rendszerekkel kommunikálni.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Jogi nyilatkozat**:
Ez a dokumentum az AI fordítási szolgáltatás, a [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével készült. Bár az pontosságra törekszünk, kérjük, vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti dokumentum az anyanyelvén tekintendő hiteles forrásnak. Fontos információk esetén professzionális emberi fordítást javasolunk. Nem vállalunk felelősséget semmilyen félreértésért vagy téves értelmezésért, amely ebből a fordításból ered.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
